<a href="https://colab.research.google.com/github/johanhoffman/DD2365_FEniCSx/blob/main/template-report-Elasticity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# About the code

In [ ]:
# Copyright (C) 2019-2026 Johan Hoffman (jhoffman@kth.se)
#
# This file is part of the course DD2365 Advanced Computation in Fluid Mechanics
# KTH Royal Institute of Technology, Stockholm, Sweden
#
# This is free software: you can redistribute it and/or modify
# it under the terms of the GNU Lesser General Public License as published by
# the Free Software Foundation, either version 3 of the License, or
# (at your option) any later version.
#
# This template is maintained by Johan Hoffman
# Please report problems to jhoffman@kth.se

# Setup environment

In [ ]:
# --- canonical: bootstrap v2 ---
import os as _os
_os.environ.setdefault("OMP_NUM_THREADS", "1")
_os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
_os.environ.setdefault("VECLIB_MAXIMUM_THREADS", "1")
_os.environ.setdefault("MKL_NUM_THREADS", "1")

import sys, os, subprocess

_on_colab = "google.colab" in sys.modules

if _on_colab:
    try:
        import gmsh
    except ImportError:
        subprocess.run(
            'wget -q "https://fem-on-colab.github.io/releases/gmsh-install.sh"'
            ' -O /tmp/gmsh-install.sh && bash /tmp/gmsh-install.sh',
            shell=True, check=True,
        )
    try:
        import dolfinx
    except ImportError:
        subprocess.run(
            'wget -q "https://fem-on-colab.github.io/releases/fenicsx-install-release-real.sh"'
            ' -O /tmp/fenicsx-install.sh && bash /tmp/fenicsx-install.sh',
            shell=True, check=True,
        )

import dolfinx
print("dolfinx version:", dolfinx.__version__)

if _on_colab:
    from google.colab import files
# --- end canonical: bootstrap ---

In [ ]:
# --- canonical: gmsh_rect_minus_circles v6 ---
import math
import numpy as np
import gmsh
from mpi4py import MPI
from dolfinx.io import gmsh as gmshio


_ALPHA = 0.66
# mshr numSamples=100 → N = floor(sqrt(100.5)) = 10 interior sample rows
_N_SAMPLE = int(math.floor(math.sqrt(100.0 + 0.5)))  # 10


def _mshr_segments(r, L, H, resolution):
    """Exact mshr auto-segment count for Circle(segments=0) in 2D.

    Source path:
      MeshGenerator.cpp: segment_granularity = 2*R_enc/resolution
      CSGGeometry.cpp:   estimate_bounding_sphere samples _N_SAMPLE×_N_SAMPLE
                         interior points; extreme corners dominate, giving
                         R_enc = ((N-1)/N) * sqrt(L²+H²) / 2
      CSGCGALDomain2D.cpp: num_segs = max(5, round(2π*r / segment_granularity))

    Verified predictions (no Steiner points since edge_len < cs_mshr):
      4×2 domain, r=0.2, resolution=32 → 10
      4×4 domain, r=0.2, resolution=32 →  8
    """
    R_enc = ((_N_SAMPLE - 1) / _N_SAMPLE) * math.sqrt(L ** 2 + H ** 2) / 2.0
    cs = 2.0 * R_enc / resolution
    return max(5, round(2.0 * math.pi * r / cs))


def gmsh_rect_minus_circles(L, H, circles, resolution, segments=32,
                             dist_min=None, dist_max=None):
    """Rectangle [0,L]×[0,H] minus regular-polygon holes.

    Parameters
    ----------
    L, H : float
        Rectangle dimensions.
    circles : list of (cx, cy, r)
        Circle centres and radii.
    resolution : int
        Mesh density parameter.  Interior mesh size:
            lc = _ALPHA * sqrt(L² + H²) / resolution
    segments : int or "mshr"
        Number of polygon sides used to approximate each circle hole.
        Default 32.  Pass ``"mshr"`` to use the legacy mshr auto-segment
        rule (max(5, round(2π·r/cs)), cs derived from bounding-sphere
        estimate), which gives 10 sides for 4×2 domains and 8 for 4×4.
    dist_min : float or None
        Threshold DistMin per hole (inner boundary of transition zone).
        Default None → 0.0 (fine zone starts at the polygon surface).
    dist_max : float or None
        Threshold DistMax per hole (outer boundary of transition zone).
        Default None → r/2 per hole (reproduces v5 behaviour exactly).

    Returns
    -------
    msh : dolfinx.mesh.Mesh
    cell_tags : dolfinx.mesh.MeshTags   (fluid domain tag = 10)

    Notes
    -----
    Vertex placement: phi_i = 2π·i/n, i = 0 … n-1 (i=0 at angle 0, CCW).

    Mesh sizing: one Distance+Threshold gmsh field per polygon transitions
    from ``edge_len = 2r·sin(π/n)`` right at the polygon boundary (prevents
    Steiner-point insertion on polygon edges) to ``lc`` at distance
    ``dist_max`` (default r/2).  No MeshSizeMax override; background field
    is the sole size control.
    """
    lc = _ALPHA * math.sqrt(L ** 2 + H ** 2) / resolution
    eps = 1e-6 * max(L, H)

    gmsh.initialize()
    gmsh.option.setNumber("General.Terminal", 0)

    gmsh.model.occ.addRectangle(0, 0, 0, L, H, tag=1)

    hole_tags = []
    segs_per_circle = []
    for cx, cy, r in circles:
        if segments == "mshr":
            n = _mshr_segments(r, L, H, resolution)
        elif isinstance(segments, int):
            n = segments
        else:
            raise ValueError(f"segments must be an int or 'mshr', got {segments!r}")
        segs_per_circle.append(n)
        pts = [
            gmsh.model.occ.addPoint(
                cx + r * math.cos(2 * math.pi * i / n),
                cy + r * math.sin(2 * math.pi * i / n),
                0,
            )
            for i in range(n)
        ]
        lines = [
            gmsh.model.occ.addLine(pts[i], pts[(i + 1) % n])
            for i in range(n)
        ]
        cl = gmsh.model.occ.addCurveLoop(lines)
        surf = gmsh.model.occ.addPlaneSurface([cl])
        hole_tags.append((2, surf))

    if hole_tags:
        gmsh.model.occ.cut([(2, 1)], hole_tags)
    gmsh.model.occ.synchronize()

    # Distance+Threshold field per polygon hole
    bg_fields = []
    for i, (cx, cy, r) in enumerate(circles):
        n = segs_per_circle[i]
        edge_len = 2 * r * math.sin(math.pi / n)

        # Identify polygon curves (not on the rectangle boundary)
        c_curves = []
        for dim, tag in gmsh.model.getEntities(1):
            xmin, ymin, _, xmax, ymax, _ = gmsh.model.getBoundingBox(dim, tag)
            if xmin <= eps or xmax >= L - eps or ymin <= eps or ymax >= H - eps:
                continue
            bounds = gmsh.model.getParametrizationBounds(dim, tag)
            mid = float(np.array(bounds).mean())
            x, y, _ = gmsh.model.getValue(dim, tag, [mid])
            if abs(math.hypot(x - cx, y - cy) - r) < 0.15 * r:
                c_curves.append(tag)

        if not c_curves:
            continue

        dm_min = 0.0 if dist_min is None else dist_min
        dm_max = r / 2.0 if dist_max is None else dist_max

        d_id = 10 + 2 * i
        t_id = 11 + 2 * i
        gmsh.model.mesh.field.add("Distance", d_id)
        gmsh.model.mesh.field.setNumbers(d_id, "CurvesList", c_curves)
        gmsh.model.mesh.field.add("Threshold", t_id)
        gmsh.model.mesh.field.setNumber(t_id, "InField", d_id)
        # SizeMin = edge_len at polygon: prevents subdivision of polygon edges.
        # SizeMax = lc at dist_max: interior mesh size.
        gmsh.model.mesh.field.setNumber(t_id, "SizeMin", edge_len)
        gmsh.model.mesh.field.setNumber(t_id, "SizeMax", lc)
        gmsh.model.mesh.field.setNumber(t_id, "DistMin", dm_min)
        gmsh.model.mesh.field.setNumber(t_id, "DistMax", dm_max)
        bg_fields.append(t_id)

    if bg_fields:
        if len(bg_fields) == 1:
            gmsh.model.mesh.field.setAsBackgroundMesh(bg_fields[0])
        else:
            min_id = 10 + 2 * len(circles)
            gmsh.model.mesh.field.add("Min", min_id)
            gmsh.model.mesh.field.setNumbers(min_id, "FieldsList", bg_fields)
            gmsh.model.mesh.field.setAsBackgroundMesh(min_id)

    # Background field is the sole size control
    gmsh.option.setNumber("Mesh.MeshSizeExtendFromBoundary", 0)
    gmsh.option.setNumber("Mesh.MeshSizeFromPoints", 0)
    gmsh.option.setNumber("Mesh.MeshSizeFromCurvature", 0)
    gmsh.option.setNumber("Mesh.Algorithm", 5)  # Delaunay

    surfaces = gmsh.model.getEntities(2)
    gmsh.model.addPhysicalGroup(2, [s[1] for s in surfaces], tag=10, name="domain")
    gmsh.model.mesh.generate(2)

    mesh_data = gmshio.model_to_mesh(gmsh.model, MPI.COMM_WORLD, rank=0, gdim=2)
    gmsh.finalize()
    return mesh_data.mesh, mesh_data.cell_tags
# --- end canonical: gmsh_rect_minus_circles ---

In [ ]:
# --- canonical: refine_cells v1 ---
import numpy as np
import dolfinx.mesh
from dolfinx.mesh import RefinementOption


def refine_cells(msh, predicate_or_mask):
    """Refine selected cells using the Plaza algorithm.

    Parameters
    ----------
    msh : dolfinx.mesh.Mesh
    predicate_or_mask : callable or array-like of bool
        Either a callable ``f(midpoints) -> bool array`` where
        ``midpoints`` has shape ``(ncells, gdim)``, or a boolean array
        of length ``num_local_cells`` that directly marks which cells
        to refine (useful when marks come from an existing DG0 field).

    Returns
    -------
    refined_msh : dolfinx.mesh.Mesh
    parent_cells : np.ndarray[np.int32]  shape (num_refined_cells,)
    parent_facets : np.ndarray[np.int8]  shape (num_refined_cells,)
        Local parent-facet index per refined cell, or -1 for interior.
    """
    tdim = msh.topology.dim
    num_cells = msh.topology.index_map(tdim).size_local

    if callable(predicate_or_mask):
        msh.topology.create_entities(1)
        msh.topology.create_connectivity(tdim, 0)
        midpoints = dolfinx.mesh.compute_midpoints(
            msh, tdim, np.arange(num_cells, dtype=np.int32)
        )
        marked = np.where(predicate_or_mask(midpoints))[0].astype(np.int32)
    else:
        mask = np.asarray(predicate_or_mask, dtype=bool)
        marked = np.where(mask[:num_cells])[0].astype(np.int32)

    if len(marked) == 0:
        n = msh.topology.index_map(tdim).size_local
        return msh, np.arange(n, dtype=np.int32), np.full(n, -1, dtype=np.int8)

    msh.topology.create_entities(1)
    msh.topology.create_connectivity(tdim, 1)
    edges = dolfinx.mesh.compute_incident_entities(msh.topology, marked, tdim, 1)
    edges = np.unique(edges).astype(np.int32)
    return dolfinx.mesh.refine(
        msh, edges, option=RefinementOption.parent_cell_and_facet
    )
# --- end canonical: refine_cells ---

In [ ]:
# --- canonical: tag_boundaries v1 ---
import numpy as np
from dolfinx.mesh import locate_entities_boundary, meshtags


def tag_boundaries(msh, L, H, eps=None):
    """Tag exterior boundary facets of a [0,L]×[0,H] rectangle with holes.

    Assigns integer tags to all exterior boundary facets:

        left=1  (x ≈ 0),   right=2 (x ≈ L),
        lower=3 (y ≈ 0),   upper=4 (y ≈ H),
        objects=5 (remaining exterior facets — circle boundaries).

    Corners are unambiguous: boundary facets are edges, not vertices, so a
    corner vertex is shared by one vertical and one horizontal edge — each
    edge belongs to exactly one group and no facet is tagged twice.

    Parameters
    ----------
    msh : dolfinx.mesh.Mesh
    L, H : float
        Rectangle dimensions.
    eps : float, optional
        Coordinate tolerance.  Default ``1e-6 * max(L, H)``.

    Returns
    -------
    dolfinx.mesh.MeshTags
        Must be recomputed whenever the mesh changes (e.g. after refinement).
    """
    if eps is None:
        eps = 1e-6 * max(L, H)

    fdim = msh.topology.dim - 1
    msh.topology.create_entities(fdim)
    msh.topology.create_connectivity(fdim, msh.topology.dim)

    left  = locate_entities_boundary(msh, fdim, lambda x: x[0] <= eps)
    right = locate_entities_boundary(msh, fdim, lambda x: x[0] >= L - eps)
    lower = locate_entities_boundary(msh, fdim, lambda x: x[1] <= eps)
    upper = locate_entities_boundary(msh, fdim, lambda x: x[1] >= H - eps)

    known = np.unique(np.concatenate([left, right, lower, upper]))
    all_bdry = locate_entities_boundary(
        msh, fdim, lambda x: np.ones(x.shape[1], dtype=bool)
    )
    objects = np.setdiff1d(all_bdry, known)

    indices = np.concatenate([left, right, lower, upper, objects]).astype(np.int32)
    values  = np.concatenate([
        np.full(len(left),    1, dtype=np.int32),
        np.full(len(right),   2, dtype=np.int32),
        np.full(len(lower),   3, dtype=np.int32),
        np.full(len(upper),   4, dtype=np.int32),
        np.full(len(objects), 5, dtype=np.int32),
    ])
    order = np.argsort(indices)
    return meshtags(msh, fdim, indices[order], values[order])
# --- end canonical: tag_boundaries ---

In [ ]:
# --- canonical: plot_helpers v3 ---
import numpy as np
import matplotlib.pyplot as plt
import basix.ufl as _bufl
from dolfinx.fem import functionspace, Function


def plot_mesh(msh, title="Mesh"):
    """Plot a 2-D triangular mesh using matplotlib triplot."""
    msh.topology.create_connectivity(msh.topology.dim, 0)
    x = msh.geometry.x
    gdm = msh.geometry.dofmaps[0]
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.triplot(x[:, 0], x[:, 1], gdm, linewidth=0.3, color="k")
    ax.set_aspect("equal")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


def _p1_scalar_space(msh):
    return functionspace(msh, ("Lagrange", 1))


def _scatter_p1_scalar(f1):
    """Scatter a P1 scalar Function values to geometry nodes.

    Returns (x, geometry_connectivity, node_values).
    """
    V = f1.function_space
    msh = V.mesh
    x = msh.geometry.x
    gdm = msh.geometry.dofmaps[0]
    ldm = V.dofmap.list
    vals = np.zeros(x.shape[0])
    vals[gdm.ravel()] = f1.x.array.real[ldm.ravel()]
    return x, gdm, vals


def plot_p1(u, title="Solution"):
    """Plot a P1 scalar Function using matplotlib tripcolor (Gouraud shading)."""
    plot_scalar(u, title=title)


def plot_scalar(f, title="Scalar field", ax=None):
    """Plot any scalar Lagrange Function via tripcolor (interpolates to P1 if needed).

    Parameters
    ----------
    f : dolfinx.fem.Function   scalar; any Lagrange degree
    title : str
    ax : matplotlib.axes.Axes or None
        If given, draw into this axes (no new figure, no plt.show()).
        If None (default), create a new figure and call plt.show().
    """
    V = f.function_space
    msh = V.mesh
    el = V.ufl_element()
    if el.degree == 1 and el.reference_value_shape == ():
        f1 = f
    else:
        f1 = Function(_p1_scalar_space(msh))
        f1.interpolate(f)
    x, gdm, vals = _scatter_p1_scalar(f1)
    _standalone = ax is None
    if _standalone:
        fig, ax = plt.subplots(figsize=(8, 3))
    tc = ax.tripcolor(x[:, 0], x[:, 1], gdm, vals, shading="gouraud")
    plt.colorbar(tc, ax=ax)
    ax.set_aspect("equal")
    ax.set_title(title)
    if _standalone:
        plt.tight_layout()
        plt.show()


def plot_vector(u, title="Vector field", quiver=True, quiver_stride=8, ax=None):
    """Plot a 2-D vector Lagrange Function as colour map of |u| + optional quiver.

    Parameters
    ----------
    u : dolfinx.fem.Function   value shape (2,); any Lagrange degree
    title : str
    quiver : bool              overlay subsampled arrows (default True)
    quiver_stride : int        take every N-th mesh node for arrows
    ax : matplotlib.axes.Axes or None
        If given, draw into this axes (no new figure, no plt.show()).
        If None (default), create a new figure and call plt.show().
    """
    msh = u.function_space.mesh
    gdim = msh.geometry.dim
    x = msh.geometry.x
    gdm = msh.geometry.dofmaps[0]
    npts = x.shape[0]

    # Interpolate to P1 vector so values align with geometry nodes
    V1v = functionspace(msh, _bufl.element("Lagrange", msh.basix_cell(), 1, shape=(gdim,)))
    u1v = Function(V1v)
    u1v.interpolate(u)

    # Scatter: for a block-size-2 P1 space, dofmap.list gives block indices
    ldm = V1v.dofmap.list      # (ncells, 3)  — block indices
    arr = u1v.x.array.real     # length npts * gdim, interleaved per block
    ux = np.zeros(npts)
    uy = np.zeros(npts)
    ux[gdm.ravel()] = arr[ldm.ravel() * gdim + 0]
    uy[gdm.ravel()] = arr[ldm.ravel() * gdim + 1]
    mag = np.sqrt(ux**2 + uy**2)

    _standalone = ax is None
    if _standalone:
        fig, ax = plt.subplots(figsize=(8, 3))
    tc = ax.tripcolor(x[:, 0], x[:, 1], gdm, mag, shading="gouraud", cmap="viridis")
    plt.colorbar(tc, ax=ax, label="|u|")
    if quiver:
        idx = np.arange(0, npts, quiver_stride)
        ax.quiver(x[idx, 0], x[idx, 1], ux[idx], uy[idx],
                  color="white", alpha=0.6, width=0.002, scale_units="xy", scale=2.0)
    ax.set_aspect("equal")
    ax.set_title(title)
    if _standalone:
        plt.tight_layout()
        plt.show()
# --- end canonical: plot_helpers ---

In [ ]:
import numpy as np
from mpi4py import MPI
from dolfinx import fem
from dolfinx.fem import (
    functionspace, Function, Constant, form,
    assemble_scalar, locate_dofs_topological,
)
from dolfinx.fem.petsc import (
    assemble_matrix, assemble_vector, apply_lifting, set_bc,
    create_matrix, create_vector,
)
from petsc4py import PETSc
import ufl
import basix.ufl as bufl
import matplotlib.pyplot as plt

# **Introduction**

Linear elasticity models the deformation of a solid body under applied loads. The displacement $d$ satisfies the equilibrium equation

$$-\nabla\cdot\sigma(d) = f$$

where $\sigma(d) = \lambda(\nabla\cdot d)I + 2\mu\,\varepsilon(d)$ is the Cauchy stress tensor and $\varepsilon(d) = \frac{1}{2}(\nabla d + \nabla d^T)$ is the linearized strain tensor. We use a P1 Galerkin discretization.

# **Method**

**Define domain and mesh**

In [ ]:
# Define rectangular domain
L = 4.0
H = 2.0

# Three-hole geometry as in Poisson (gmsh v6 defaults, segments=32)
circles = [
    (1.5, 0.25 * H, 0.2),
    (0.5, 0.50 * H, 0.2),
    (2.0, 0.75 * H, 0.2),
]

resolution = 32
msh, _ = gmsh_rect_minus_circles(L, H, circles, resolution)
facet_tags = tag_boundaries(msh, L, H)

# Local mesh refinement (no_levels=0 → no refinement)
no_levels = 0
for _ in range(no_levels):
    msh, _, _ = refine_cells(
        msh,
        lambda pts: np.sqrt((pts[:, 0] - 1.5)**2 + (pts[:, 1] - 0.5)**2) < 1.0,
    )
    facet_tags = tag_boundaries(msh, L, H)

tdim = msh.topology.dim
fdim = tdim - 1
n_cells = msh.topology.index_map(tdim).size_local

plot_mesh(msh)

**Define finite element approximation space**

In [ ]:
# Generate finite element space (P1 vector for displacement)
P1v = bufl.element("Lagrange", msh.basix_cell(), 1, shape=(2,))
V = functionspace(msh, P1v)

# Define trial and test functions
u = ufl.TrialFunction(V)
v = ufl.TestFunction(V)

# Displacement solution
d = Function(V, name="displacement")

**Define boundary conditions**

In [ ]:
# Define boundary conditions
def _bc(val, sub_idx, sub_V, facets):
    f = Function(sub_V); f.x.array[:] = val
    dofs = fem.locate_dofs_topological((V.sub(sub_idx), sub_V), fdim, facets)
    return fem.dirichletbc(f, dofs, V.sub(sub_idx))

V0, _ = V.sub(0).collapse()
V1, _ = V.sub(1).collapse()

left_f  = facet_tags.find(1)
right_f = facet_tags.find(2)
lower_f = facet_tags.find(3)
upper_f = facet_tags.find(4)
obj_f   = facet_tags.find(5)

# Outer boundary: fully fixed (u_x=0, u_y=0 on all four walls)
bcu = [
    _bc(0., 0, V0, left_f),  _bc(0., 1, V1, left_f),
    _bc(0., 0, V0, right_f), _bc(0., 1, V1, right_f),
    _bc(0., 0, V0, upper_f), _bc(0., 1, V1, upper_f),
    _bc(0., 0, V0, lower_f), _bc(0., 1, V1, lower_f),
    # Objects: prescribed displacement u_x=0.5, u_y=0
    _bc(0.5, 0, V0, obj_f),  _bc(0., 1, V1, obj_f),
]

# **Results**

**Define material parameters**

In [ ]:
# Material parameters (exactly as legacy)
f_body = Constant(msh, PETSc.ScalarType((0.0, 0.0)))

dim = 2
E     = 1.0e10
nu_p  = 0.3           # Poisson ratio (not flow viscosity)
mu_e  = E * 0.5 / (1.0 + nu_p)
lam   = nu_p * E / ((1.0 + nu_p) * (1.0 - 2.0 * nu_p))

def epsilon(u_fn):
    return 0.5 * (ufl.grad(u_fn) + ufl.grad(u_fn).T)

def sigma(u_fn):
    return lam * ufl.div(u_fn) * ufl.Identity(dim) + 2.0 * mu_e * epsilon(u_fn)

**Define variational problem and solve**

In [ ]:
# --- canonical: ksp_helpers v1 ---
from petsc4py import PETSc


def make_ksp(A, ksp_type="bcgs", pc_type="ilu", amg=False, rtol=1e-6, atol=1e-14, max_it=500):
    ksp = PETSc.KSP().create(A.comm)
    ksp.setOperators(A)
    ksp.setType(ksp_type)
    pc = ksp.getPC()
    if amg:
        pc.setType("hypre")
        pc.setHYPREType("boomeramg")
    else:
        pc.setType(pc_type)
    ksp.setTolerances(rtol=rtol, atol=atol, max_it=max_it)
    ksp.setFromOptions()
    return ksp


def solve_checked(ksp, b, x, name):
    ksp.solve(b, x)
    reason = ksp.getConvergedReason()
    iters = ksp.getIterationNumber()
    if reason < 0:
        raise RuntimeError(
            f"KSP '{name}' diverged: reason={reason}, iterations={iters}"
        )
# --- end canonical: ksp_helpers ---

In [ ]:
# Variational problem on residual form (exactly as legacy)
residual = ufl.inner(sigma(u), epsilon(v)) * ufl.dx - ufl.inner(f_body, v) * ufl.dx
a_e = form(ufl.lhs(residual))
L_e = form(ufl.rhs(residual))

# Assemble system
A_e = create_matrix(a_e); b_e = create_vector(V)
assemble_matrix(A_e, a_e, bcs=bcu); A_e.assemble()
with b_e.localForm() as loc: loc.set(0.0)
assemble_vector(b_e, L_e)
apply_lifting(b_e, [a_e], bcs=[bcu])
b_e.ghostUpdate(addv=PETSc.InsertMode.ADD, mode=PETSc.ScatterMode.REVERSE)
set_bc(b_e, bcu)

# Solve: CG + BoomerAMG (SPD system; legacy used bicgstab/default)
ksp = make_ksp(A_e, ksp_type="cg", amg=True, rtol=1e-8, max_it=1000)
solve_checked(ksp, b_e, d.x.petsc_vec, "ksp")
d.x.scatter_forward()
set_bc(d.x.petsc_vec, bcu); d.x.scatter_forward()
print(f"KSP converged in {ksp.getIterationNumber()} iterations")

**Plot displacement**

In [ ]:
plot_vector(d, title="Displacement")

**Move mesh and plot deformed geometry**

In [ ]:
# Move mesh by displacement d (same dof->geometry mapping as NS-ALE)
def _move_mesh_by_d(d_fn):
    gdm = msh.geometry.dofmaps[0]
    ldm = V.dofmap.list
    d_arr = d_fn.x.array.real
    dx = np.zeros(msh.geometry.x.shape[0])
    dy = np.zeros(msh.geometry.x.shape[0])
    dx[gdm.ravel()] = d_arr[ldm.ravel() * 2 + 0]
    dy[gdm.ravel()] = d_arr[ldm.ravel() * 2 + 1]
    msh.geometry.x[:, 0] += dx
    msh.geometry.x[:, 1] += dy

_move_mesh_by_d(d)
plot_mesh(msh, title="Deformed mesh")

**QoIs**

In [ ]:
norm_d = float(msh.comm.allreduce(
    assemble_scalar(form(ufl.inner(d, d)*ufl.dx)), op=MPI.SUM
))**0.5

# Min cell volume on deformed mesh
DG0s = functionspace(msh, bufl.element("DG", msh.basix_cell(), 0))
vol_fn = Function(DG0s)
vol_fn.interpolate(
    fem.Expression(ufl.CellVolume(msh), DG0s.element.interpolation_points)
)
min_vol = float(vol_fn.x.array.min())

print(f"||d||  = {norm_d:.6f}")
print(f"cells  = {n_cells}")
print(f"min_cell_vol (deformed) = {min_vol:.6g}")
print(f"QOI norm_d = {norm_d:.6g}")
print(f"QOI min_vol = {min_vol:.6g}")

# **Discussion**

The P1 Galerkin discretization of the elasticity equations gives a symmetric positive definite (SPD) linear system, which is solved efficiently with Conjugate Gradient preconditioned by BoomerAMG. The deformed mesh is obtained by adding the displacement field $d$ to the initial mesh coordinates, using the same DOF-to-geometry mapping as in the NS-ALE notebook.